### This notebook shows how to use PyGlaucoMetrics module

In [2]:
import PyVisualFields
print(PyVisualFields.__file__)
print(PyVisualFields.__version__)

E:\Partners HealthCare Dropbox\Mohammad Eslami\GithubRepos\PyVisualField\src\PyVisualFields\__init__.py
2.0.8


In [3]:
from PyVisualFields import visualFields
from PyVisualFields.utils import canonicalize_vf_df
from PyVisualFields.utils import vf_blocks, missing_blocks
from PyVisualFields.utils import compute_missing_blocks
from PyVisualFields.utils import print_vf_summary, investigate_vf_df
from PyVisualFields import vfprogression
import pandas as pd

df_VFs = canonicalize_vf_df(vfprogression.data_vfseries())

print('---> blocks',vf_blocks(df_VFs))
print('---> missed blocks', missing_blocks(df_VFs))

#### you can investigate the available information in the dataframe
print_vf_summary(df_VFs) # this function will print

---> blocks {'sens': True, 'td': True, 'pd': True, 'tdp': True, 'pdp': True}
---> missed blocks []
=== VF Data Summary ===
Rows: 20
Columns: 286

Pointwise blocks:
  ✓ sensitivity (54 locations)
  ✓ total_deviation (54 locations)
  ✓ pattern_deviation (54 locations)
  ✓ total_deviation_probability (54 locations)
  ✓ pattern_deviation_probability (54 locations)

Global indices:
  ✓ md
  ✓ mdprob
  ✓ psd
  ✓ psdprob
  ✓ ght
  ✓ vfi
  ✗ msens
  ✗ ssens
  ✗ tmd
  ✗ tsd
  ✗ pmd
  ✗ gh

Metadata:
  ✗ patientid
  ✓ eyeid
  ✗ date
  ✓ age
  ✓ yearsfollowed
  ✓ duration


In [4]:

# glaucoma metrics depend on TD, Tdp, PD, and PDP values.
# also sometimes GHT, PSD, MD, etc.
# so let's compute them if not available 
# GHT can not be calculated
import pandas as pd
from PyVisualFields.utils import compute_missing_blocks
from PyVisualFields import visualFields

print(df_VFs.shape)
df_VFs = compute_missing_blocks(df_VFs)
print('---> blocks',vf_blocks(df_VFs))
print('---> missed blocks', missing_blocks(df_VFs))
print(df_VFs.shape)

df_gi = visualFields.getgl(df_VFs) 
combined_df = pd.concat([df_VFs, df_gi], axis=1)
final_df = combined_df.loc[:, ~combined_df.columns.duplicated()]

(20, 286)
---> blocks {'sens': True, 'td': True, 'pd': True, 'tdp': True, 'pdp': True}
---> missed blocks []
(20, 286)
==> py_getgl: Already available global indices: ['psd', 'vfi']
==> py_getgl: missing global indices to compute: ['msens', 'ssens', 'tmd', 'tsd', 'pmd', 'gh']


In [5]:
final_df.head()

,eyeid,nvisit,yearsfollowed,distprev,age,righteye,duration,falsenegrate,falseposrate,malfixrate,...,pdp51,pdp52,pdp53,pdp54,msens,ssens,tmd,tsd,pmd,gh
0,1.0,1,0.00,NaN,64.98,1,287,0.0,0.00,0.071429,...,1.0,1.0,1.0,1.0,29.000000,2.283788,-0.621837,1.537917,-1.854798,1.0
1,1.0,2,0.73,0.73,65.71,1,295,0.0,0.01,0.142857,...,1.0,1.0,1.0,1.0,29.346154,2.383642,-0.309503,1.574159,-1.489677,1.0
2,1.0,3,1.77,1.04,66.75,1,291,0.0,0.01,0.062500,...,1.0,1.0,1.0,1.0,29.461538,2.287747,-0.189323,1.899977,-1.968863,2.0
3,1.0,4,2.78,1.01,67.76,1,347,0.0,0.02,0.066667,...,1.0,1.0,1.0,1.0,27.576923,2.538690,-1.843116,1.862786,-1.803049,0.0
4,1.0,5,4.01,1.23,68.99,1,309,0.0,0.01,0.000000,...,1.0,1.0,1.0,1.0,27.500000,2.866302,-1.986323,2.610211,-2.922875,1.0


In [6]:
from PyVisualFields import PyGlaucoMetrics 

# HAP2 criteria
Pattern Deviation Probabilities


GHT “outside normal limits”  </br>
or </br>
cluster of 3 PDP points P < 5%, 1 of which must be < 1% </br>
or </br>
PSD at P < 5%

In [7]:
# classify the visual fields for glaucoma based on HAP2
# Determined class will be shown correspondingly in columns: 'HAP2_clf'

# Hap2 criteria can consider the PSD probability (psdp) and GHT too. but optional
# so we optionally added them, but nan for this example
# Hap2 Part I: Minimum criteria for diagnosing acquired glaucomatous damage
# https://optometricglaucomasociety.org/wp-content/uploads/2024/01/Clinical-Decisions-in-Glaucoma_compressed.pdf
### page 53

df_diag_HAP2 = PyGlaucoMetrics.Fn_HAP2(final_df)
print(df_diag_HAP2[['l1', 'l2', 'HAP2_clf', 'HAP2_reason']].head())

   l1  l2 HAP2_clf                                        HAP2_reason
0  26  28   Non-GL                                                   
1  26  25   Non-GL                                                   
2  28  27   Non-GL                                                   
3  24  25       GL  GHT outside; cluster of 3 PDP points < 5% with...
4  26  26       GL  GHT outside; cluster of 3 PDP points < 5% with...


In [8]:
## HAP 2 part II: classification of the severity of visual field defects
# Part II – Classification of glaucomatous visual field defects
# It needs sensitivity, and pdp, and MD
# The severity results are shown in severity(HAP2_part2)
# Early, Moderate, Severe
# https://optometricglaucomasociety.org/wp-content/uploads/2024/01/Clinical-Decisions-in-Glaucoma_compressed.pdf
### page 64

df_diag_HAP2withPartII = PyGlaucoMetrics.Fn_HAP2_part2(df_diag_HAP2)
print(df_diag_HAP2withPartII[['l1', 'l2', 'HAP2_clf', 'severity(HAP2_part2)']].head())

   l1  l2 HAP2_clf severity(HAP2_part2)
0  26  28   Non-GL                Early
1  26  25   Non-GL                Early
2  28  27   Non-GL                Early
3  24  25       GL                Early
4  26  26       GL                Early


# UKGTS criteria
Total Deviation and Total Deviation Probabilities


cluster of 2 TDP points < 1% </br>
or </br>
cluster of 3 TDP points < 5% </br>
or </br>
cluster of 2 TD points with 10 dB difference across the nasal horizontal midline

In [9]:
# classify the visual fields for glaucoma based on UKGTS
# Determined class will be shown correspondingly in columns: 'UKGTS_clf',

df_diag_UKGTS = PyGlaucoMetrics.Fn_UKGTS(final_df)
print(df_diag_UKGTS[['l1', 'l2', 'UKGTS_clf', 'UKGTS_reason']].head())

   l1  l2 UKGTS_clf                                       UKGTS_reason
0  26  28    Non-GL                                                   
1  26  25    Non-GL                                                   
2  28  27    Non-GL                                                   
3  24  25        GL  Cluster of 2 TDP points < 1%; Cluster of 3 TDP...
4  26  26        GL  Cluster of 2 TDP points < 1%; Cluster of 3 TDP...


# LoGTS criteria
Total Deviation

cluster of 2 TD points < −10 dB </br>
or </br>
cluster of 3 TD points < −8 dB

In [10]:
# classify the visual fields for glaucoma based on LoGTS
# Determined class will be shown correspondingly in columns: 'LoGTS_clf',
df_diag_LoGTS = PyGlaucoMetrics.Fn_LoGTS(final_df)
print(df_diag_LoGTS[['l1', 'l2', 'LoGTS_clf', 'LoGTS_reason']].head())

   l1  l2 LoGTS_clf LoGTS_reason
0  26  28    Non-GL             
1  26  25    Non-GL             
2  28  27    Non-GL             
3  24  25    Non-GL             
4  26  26    Non-GL             


# Foster criteria:
Needs pattern deviation probabilities (PDPs) and GHT. Criteria:
    
GHT outside normal limits </br>
AND </br>
cluster of 3 PDP points  < 5%

In [11]:
# classify the visual fields for glaucoma based on Foster
# Determined class will be shown correspondingly in columns: 'Foster_clf',

# Foster needs GHT, so if it is not available, you will receive an error

df_diag_Foster = PyGlaucoMetrics.Fn_Foster(final_df)
print(df_diag_Foster[['l1', 'l2', 'Foster_clf', 'Foster_reason']].head())

   l1  l2 Foster_clf                                 Foster_reason
0  26  28     Non-GL                                              
1  26  25     Non-GL                                              
2  28  27     Non-GL                                              
3  24  25         GL  GHT outside AND cluster of 3 PDP points < 5%
4  26  26         GL  GHT outside AND cluster of 3 PDP points < 5%



# Kangs criteria
Total Deviation

criterion 1: cluster of 3 contiguous points with TD < -5 dB  </br>
or </br>
criterion 2: cluster of 2 contiguous points with TD < -10 dB

    

In [12]:
# classify the visual fields for glaucoma based on Kang
# Determined class will be shown correspondingly in columns:  'Kangs_clf.'
df_diag_Kangs = PyGlaucoMetrics.Fn_Kangs(final_df)
print(df_diag_Kangs[['l1', 'l2', 'Kangs_clf', 'Kangs_reason']].head())

   l1  l2 Kangs_clf                         Kangs_reason
0  26  28    Non-GL                                     
1  26  25    Non-GL                                     
2  28  27    Non-GL                                     
3  24  25    Non-GL                                     
4  26  26        GL  Cluster of 3 points with TD < -5 dB
